# BigMart Modeling and Final Prediction

This notebook handles the full modeling stage after preparation.

The workflow is simple: load the train/validation split prepared in notebook 1, compare a few baseline regression models, lightly tune the best one, retrain the final model on the full prepared training set, and use the prepared test set only at the end for final predictions.

Every model is wrapped in a single scikit-learn `Pipeline` (preprocessing + model). That way the scaler and encoder are always fitted only on the data the model is trained on, including inside each cross-validation fold, and the final model reuses exactly the preprocessing that was validated.

## 1. Imports

Only the libraries needed for modeling and final prediction are used here.

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 2. Load the Prepared Training Data

This notebook starts from the prepared training file created in notebook 1.

In [2]:
prepared_train = pd.read_csv("prepared_train.csv")

print("Prepared train shape:", prepared_train.shape)
prepared_train.head()

Prepared train shape: (8523, 16)


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Category,Outlet_Age,Item_Visibility_Mean,Visibility_Ratio
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Food,14,0.017578,0.912912
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,4,0.019208,1.003670
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Food,14,0.020709,0.809327
3,FDX07,19.20,Regular,0.022911,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,Food,15,0.022911,1.000000
4,NCD19,8.93,Non-Edible,0.016164,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,26,0.016164,1.000000


## 3. Feature Setup and Train-Validation Split

The train/validation split is made in notebook 1, on the raw data, **before** cleaning.
That way the validation rows were cleaned using only statistics from the training part (the same way the test set is), so the validation scores are not inflated by leakage.
The test data is not used for model comparison or tuning.

In [3]:
target = "Item_Outlet_Sales"
drop_columns = ["Item_Identifier", "Outlet_Establishment_Year"]

prepared_train_part = pd.read_csv("prepared_train_part.csv")
prepared_valid_part = pd.read_csv("prepared_valid_part.csv")

X_train = prepared_train_part.drop(columns=drop_columns + [target])
y_train = prepared_train_part[target]
X_valid = prepared_valid_part.drop(columns=drop_columns + [target])
y_valid = prepared_valid_part[target]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (6818, 13)
X_valid: (1705, 13)
y_train: (6818,)
y_valid: (1705,)


In [4]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


def make_model_pipeline(model) -> Pipeline:
    """Preprocessing + model in one Pipeline, so preprocessing is always fitted
    only on the rows the model is trained on."""
    return Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", model)
    ])


n_prepared_features = clone(preprocessor).fit(X_train).get_feature_names_out().shape[0]
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Features after preprocessing:", n_prepared_features)

Numeric features: 6
Categorical features: 7
Features after preprocessing: 48


## 4. Baseline Regression Models

A small set of baseline models is compared on the same split, each with the same preprocessing.

SVR is sensitive to the scale of the target: sales range from about 33 to 13,000, and with its default settings (`C=1`, `epsilon=0.1`) SVR can barely fit values that large. To give it a fair chance, its target is standardized with `TransformedTargetRegressor`, and predictions are converted back to the original sales scale before scoring.

In [5]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200, n_jobs=1),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "SVR (scaled target)": TransformedTargetRegressor(regressor=SVR(), transformer=StandardScaler())
}

results = []
trained_models = {}

for model_name, model in models.items():
    pipeline = make_model_pipeline(model)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_valid)

    results.append({
        "Model": model_name,
        "MAE": round(mean_absolute_error(y_valid, predictions), 2),
        "RMSE": round(root_mean_squared_error(y_valid, predictions), 2),
        "R2": round(r2_score(y_valid, predictions), 4)
    })

    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(by=["RMSE", "MAE"]).reset_index(drop=True)
results_df

C:\Users\samir\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\samir\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\samir\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\samir\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\samir\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

,Model,MAE,RMSE,R2
0,SVR (scaled target),719.49,1035.03,0.6058
1,HistGradientBoostingRegressor,729.97,1051.22,0.5934
2,Random Forest Regressor,744.10,1069.02,0.5795
3,Linear Regression,791.59,1069.15,0.5794


## 5. Light Tuning of HistGradientBoostingRegressor

With its target scaled, SVR is now the strongest *untuned* baseline, with HistGradientBoostingRegressor close behind.
HistGradientBoostingRegressor is the model tuned here because it trains quickly and has a small, well-understood set of hyperparameters. After tuning it beats the SVR baseline on the validation set (see the comparison below). Tuning SVR (`C`, `epsilon`, `gamma`) would be a fair next experiment.
The search is kept compact and RMSE is used as the main selection metric.

The grid search gets the whole pipeline and the unprocessed `X_train`. In each cross-validation fold, the scaler and encoder are fitted only on that fold's training rows, so the held-out fold never influences its own preprocessing. The grid keys get a `model__` prefix because they refer to the `model` step of the pipeline.

In [6]:
param_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [15, 31],
    "model__min_samples_leaf": [20, 40],
    "model__l2_regularization": [0.0, 0.1]
}

hgb_grid = GridSearchCV(
    estimator=make_model_pipeline(HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
    verbose=0
)

hgb_grid.fit(X_train, y_train)

best_params = hgb_grid.best_params_
print("Best parameters:")
best_params

Best parameters:


{'model__l2_regularization': 0.0,
 'model__learning_rate': 0.05,
 'model__max_iter': 100,
 'model__max_leaf_nodes': 15,
 'model__min_samples_leaf': 40}

In [7]:
baseline_hgb = trained_models["HistGradientBoostingRegressor"]
tuned_hgb = hgb_grid.best_estimator_

y_pred_baseline = baseline_hgb.predict(X_valid)
y_pred_tuned = tuned_hgb.predict(X_valid)

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline HistGradientBoostingRegressor",
        "MAE": round(mean_absolute_error(y_valid, y_pred_baseline), 2),
        "RMSE": round(root_mean_squared_error(y_valid, y_pred_baseline), 2),
        "R2": round(r2_score(y_valid, y_pred_baseline), 4)
    },
    {
        "Model": "Tuned HistGradientBoostingRegressor",
        "MAE": round(mean_absolute_error(y_valid, y_pred_tuned), 2),
        "RMSE": round(root_mean_squared_error(y_valid, y_pred_tuned), 2),
        "R2": round(r2_score(y_valid, y_pred_tuned), 4)
    }
])

comparison_df

,Model,MAE,RMSE,R2
0,Baseline HistGradientBoostingRegressor,729.97,1051.22,0.5934
1,Tuned HistGradientBoostingRegressor,715.94,1024.34,0.6139


In [8]:
rmse_improvement = round(comparison_df.loc[0, "RMSE"] - comparison_df.loc[1, "RMSE"], 2)
mae_improvement = round(comparison_df.loc[0, "MAE"] - comparison_df.loc[1, "MAE"], 2)
r2_improvement = round(comparison_df.loc[1, "R2"] - comparison_df.loc[0, "R2"], 4)

print("RMSE improvement:", rmse_improvement)
print("MAE improvement:", mae_improvement)
print("R2 improvement:", r2_improvement)

RMSE improvement: 26.88
MAE improvement: 14.03
R2 improvement: 0.0205


## 6. Retrain the Final Model on the Full Prepared Training Data

Since the tuned model performs better, it is retrained on the full prepared training dataset.

The final model is a `clone` of the tuned pipeline: the same preprocessing and the same best parameters, but unfitted, so it can be refit on all training rows. Because nothing is written out a second time, the final model cannot drift away from the configuration that was validated.

In [9]:
X_full = prepared_train.drop(columns=drop_columns + [target])
y_full = prepared_train[target]

final_model = clone(hgb_grid.best_estimator_)
final_model.fit(X_full, y_full)

print("Final model trained on full prepared training data.")
print("Final model parameters:", {k: v for k, v in final_model.get_params().items() if k in best_params})

Final model trained on full prepared training data.
Final model parameters: {'model__l2_regularization': 0.0, 'model__learning_rate': 0.05, 'model__max_iter': 100, 'model__max_leaf_nodes': 15, 'model__min_samples_leaf': 40}


## 7. Use the Prepared Test Data Only at the End

The prepared test file is loaded only now, after model comparison and tuning are finished.

In [10]:
prepared_test = pd.read_csv("prepared_test.csv")
X_test = prepared_test.drop(columns=drop_columns)

print("Prepared test shape:", prepared_test.shape)
print("Test features shape:", X_test.shape)

Prepared test shape: (5681, 15)
Test features shape: (5681, 13)


## 8. Generate Final Predictions

The final model is used to predict `Item_Outlet_Sales` for the prepared test data.

In [11]:
final_predictions = final_model.predict(X_test)

prediction_df = pd.DataFrame({
    "Item_Identifier": prepared_test["Item_Identifier"],
    "Outlet_Identifier": prepared_test["Outlet_Identifier"],
    "Item_Outlet_Sales": final_predictions
})

prediction_df.head()

,Item_Identifier,Outlet_Identifier,Item_Outlet_Sales
0,FDW58,OUT049,1689.925855
1,FDW14,OUT017,1353.109423
2,NCN55,OUT010,572.311490
3,FDQ58,OUT017,2524.257805
4,FDY38,OUT027,6071.611114


In [12]:
prediction_df.to_csv("bigmart_final_predictions.csv", index=False)
print("Saved bigmart_final_predictions.csv")

Saved bigmart_final_predictions.csv
